# 06 Attention 与 Transformer 基础

## 本节目标

本节学习 Attention 的基本计算过程，并用一个小型 Transformer Block 观察输入输出形状。

你将完成：

- 理解 Query、Key、Value 的角色。
- 手写 scaled dot-product attention。
- 可视化 attention weights。
- 使用 `nn.MultiheadAttention`。
- 理解 Transformer Block 中的残差连接、LayerNorm 和前馈网络。

## 背景问题

RNN/LSTM 按时间顺序处理序列，远距离信息需要一步步传递。Attention 提供了另一种思路：让一个位置直接和其他位置计算相关性，再按权重汇总信息。

本节关注的问题是：模型如何决定“当前 token 应该关注哪些 token”？

## 核心概念

- **Query (Q)**：当前位置发出的查询。
- **Key (K)**：每个位置用于被匹配的表示。
- **Value (V)**：被加权汇总的信息。
- **Attention weights**：相关性分数经过 softmax 后得到的权重。
- **Multi-head Attention**：从多个子空间并行观察关系。
- **Transformer Block**：注意力、残差连接、归一化和前馈网络的组合。

这里的关键不是背公式，而是看懂 `QK^T -> softmax -> 加权 V` 这条链路。

## 最小代码示例：构造序列表示

下面构造一个很小的序列张量：batch 为 1，序列长度为 4，每个位置有 3 维表示。

In [ ]:
import torch
from torch import nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

x = torch.tensor([[[1.0, 0.0, 0.5],
                   [0.8, 0.2, 0.4],
                   [0.0, 1.0, 0.3],
                   [0.1, 0.9, 0.2]]])

print("x shape:", x.shape)

## 最小代码示例：手写 scaled dot-product attention

公式如下：

```text
Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V
```

缩放项 `sqrt(d_k)` 可以避免点积值过大导致 softmax 过于尖锐。

In [ ]:
def scaled_dot_product_attention(q, k, v):
    d_k = q.size(-1)
    scores = q @ k.transpose(-2, -1) / np.sqrt(d_k)
    weights = torch.softmax(scores, dim=-1)
    output = weights @ v
    return output, weights, scores


attn_output, attn_weights, attn_scores = scaled_dot_product_attention(x, x, x)

print("scores shape:", attn_scores.shape)
print("weights shape:", attn_weights.shape)
print("output shape:", attn_output.shape)
print(attn_weights[0])

## 实验观察：注意力权重

权重矩阵中的每一行对应一个 query 位置，每一列对应一个 key 位置。每一行加起来应该接近 1。

In [ ]:
print("row sums:", attn_weights[0].sum(dim=-1))

plt.figure(figsize=(4, 3.5))
plt.imshow(attn_weights[0].detach().numpy(), cmap="viridis")
plt.colorbar(label="weight")
plt.xlabel("key position")
plt.ylabel("query position")
plt.title("Attention weights")
plt.tight_layout()
plt.show()

## 最小代码示例：加入 mask 的直观理解

在一些序列任务中，模型不能看到未来位置。mask 可以把不允许关注的位置分数设为很小，让 softmax 后权重接近 0。

In [ ]:
scores = attn_scores.clone()
seq_len = scores.size(-1)
future_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()

masked_scores = scores.masked_fill(future_mask, float("-inf"))
masked_weights = torch.softmax(masked_scores, dim=-1)

print("masked attention weights:")
print(masked_weights[0])

## 完整实验：使用 MultiheadAttention

PyTorch 提供 `nn.MultiheadAttention`。当设置 `batch_first=True` 时，输入输出都是 `[batch, seq_len, embed_dim]`。

In [ ]:
embed_dim = 8
num_heads = 2

mha = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
tokens = torch.randn(2, 5, embed_dim)

attn_out, weights = mha(tokens, tokens, tokens, need_weights=True)

print("tokens:", tokens.shape)
print("attn_out:", attn_out.shape)
print("weights:", weights.shape)

## 完整实验：小型 Transformer Block

一个基础 Transformer Block 通常包含：

1. Multi-head Attention
2. 残差连接
3. LayerNorm
4. 前馈网络
5. 再一次残差连接和 LayerNorm

残差连接要求输入输出 shape 一致。

In [ ]:
class TinyTransformerBlock(nn.Module):
    def __init__(self, embed_dim=8, num_heads=2, ff_dim=16):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim),
        )
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        attn_out, weights = self.attn(x, x, x, need_weights=True)
        x = self.norm1(x + attn_out)
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x, weights


block = TinyTransformerBlock(embed_dim=8, num_heads=2)
out, block_weights = block(tokens)

print("input:", tokens.shape)
print("output:", out.shape)
print("weights:", block_weights.shape)

## 最小代码示例：位置编码的动机

Attention 本身主要看 token 之间的相似度，并不天然知道顺序。实际 Transformer 通常会加入位置信息。下面只是演示位置编码的形状。

In [ ]:
def sinusoidal_position_encoding(seq_len, dim):
    positions = torch.arange(seq_len).unsqueeze(1)
    div_terms = torch.exp(torch.arange(0, dim, 2) * (-np.log(10000.0) / dim))
    pe = torch.zeros(seq_len, dim)
    pe[:, 0::2] = torch.sin(positions * div_terms)
    pe[:, 1::2] = torch.cos(positions * div_terms)
    return pe


pe = sinusoidal_position_encoding(seq_len=5, dim=8)
print(pe.shape)
print(pe[:2])

## 实验观察

从运行结果可以看到：

- attention weights 的每一行都可以看成一个分布。
- `MultiheadAttention` 保持输入输出 shape 一致。
- Transformer Block 可以堆叠，是因为每个 block 输入输出维度相同。
- 位置编码用于补充 token 的顺序信息。

这个实验不训练模型，重点是理解计算链路和 shape。

## 常见错误

1. **softmax 维度写错**：通常应在 key 维度上归一化。  
2. **`embed_dim` 不能被 `num_heads` 整除**：多头注意力需要平分维度。  
3. **batch-first 格式混淆**：确认输入是 `[batch, seq, dim]` 还是 `[seq, batch, dim]`。  
4. **残差连接 shape 不一致**：相加前两个张量形状必须相同。  
5. **忽略位置信息**：只用 attention 时，顺序信息需要额外注入。

调试时优先打印 scores、weights、attention output 和 block output 的 shape。

## 概念问答

**Q：Q、K、V 一定来自同一个输入吗？**  
A：自注意力中通常来自同一个输入；交叉注意力中 Query 和 Key/Value 可以来自不同来源。

**Q：Attention 权重能完全解释模型吗？**  
A：不能。它提供一个观察角度，但模型输出还受到投影层、前馈层和训练目标影响。

**Q：为什么要多头？**  
A：多个 head 可以从不同子空间学习关系，再把信息合并。

**Q：Transformer 为什么容易并行？**  
A：自注意力可以同时计算多个位置之间的关系，不必像 RNN 那样严格逐步递推。

## 小结

Attention 可以理解为基于相关性的加权汇总。理解 Q/K/V、权重矩阵、mask、多头注意力和残差结构，是继续学习完整 Transformer 的基础。